### Assignment 1 code to run

In [1]:
import math
# -----------------------------
# Parameters
# -----------------------------
gamma = 0.9
lam=0.5 # used in poisson distribution
xi1, xi2 = 5, 7

# Location states
DEPOT = 0
AT_1 = 1   # engineer is @ machine 1
AT_2 = 2   # engineer is @ machine 2
REP_1 = 3   # 1 unit of remaining time for corrective maintenance @1. Note that this state only exists when x1=xi1. 
REP_2 = 4   # 1 unit of remaining time for corrective maintenance @2. Note that this state only exists when x2=xi2. 

# -----------------------------
# State space
# -----------------------------

# NOTE: we keep only states that are possible

states = []

for x1 in range(xi1 + 1):
    for x2 in range(xi2 + 1):

        # always allowed locations
        for l in [DEPOT, AT_1, AT_2]:
            states.append((x1, x2, l))

        # corrective maintenance state for machine 1
        if x1 == xi1:
            states.append((x1, x2, REP_1))

        # corrective maintenance state for machine 2
        if x2 == xi2:
            states.append((x1, x2, REP_2))


# -----------------------------
# Poisson probabilities
# -----------------------------
def poisson_pmf(y,lam, max_y):
    """
    Returns the y-th element of the list:
    [P(X=0), P(X=1), ..., P(X=max_y-1), P(X>=max_y)]
    with X ~ Poisson(lam)
    """

    probs = []

    if max_y>0:
        # P(X=0)
        p = math.exp(-lam)
        probs.append(p)

        # Compute P(X=k) recursively
        for k in range(1, max_y):
            p = p * lam / k
            probs.append(p)

        # Tail probability
        tail = 1.0 - sum(probs)
        probs.append(tail)
    elif max_y==0:
        probs=[1]
    return probs[y]

# -----------------------------
# Action space
# -----------------------------
#Note that this is further refined below when considering the possible acitons
ACTIONS = ["nothing", "travel_1", "travel_2", "travel_depot",
           "maintain_1", "maintain_2", "continue maintenance"]


"""
The action "continue maintenance" is a pseudo-action to only indicate that we are forced to continue the corrective maintenance for a second period of time and should not impact the results        
"""    

# -----------------------------
# Feasible actions
# -----------------------------

# used in e-gridy!!!

def feasible_actions(state): 
    # based on a state gets me a function of feasible actions! 
    x1, x2, l = state
    
    acts = []
    
    # forced continuation during repair
    if l in (REP_1, REP_2):
        acts = ["continue maintenance"]

    elif l == DEPOT:
        if x2 == xi2 and x1 == xi1:
            acts = ["travel_1", "travel_2"]
        elif x2 == xi2 and x1 < xi1:
            acts = ["travel_2"]
        elif x1 == xi1 and x2 < xi2:
            acts = ["travel_1"]
        else:
            acts = ["nothing", "travel_1", "travel_2"]

    elif l == AT_1:
        if x1 == xi1:
            acts = ["maintain_1"]
        elif x1 < xi1 and x2 == xi2: 
            acts = ["travel_depot"]
        else:
            acts = ["travel_depot", "maintain_1","nothing"]  
        

    elif l == AT_2:
        if x2 == xi2:
            acts = ["maintain_2"]
        elif x2 < xi2 and x1 == xi1: 
            acts = ["travel_depot"]
        else:
            acts = ["travel_depot", "maintain_2","nothing"] 

    return acts

# -----------------------------
# Cost
# -----------------------------
def cost(state, action):
    x1, x2, l = state

    c = 0

    if action == "maintain_1":
        if x1 < xi1:
            c += 1 
        else:
            c += 5 
                
    if action == "maintain_2":
        if x2 < xi2:
            c += 1 
        else:
            c += 5 

    # unavailability cost
    if x1 == xi1 :
        c += 1
    if x2 == xi2 :
        c += 1

    return c


# -----------------------------
# Transitions
# -----------------------------
def transitions(state, action):
    x1, x2, l = state
    trans = {}

    # ---------------- forced repair completion
    if l == REP_1:
        # in the next unit, maintenance completed → machine 1 becomes healthy
        for y in range(xi2-x2+1):
            p = poisson_pmf(y,lam, xi2-x2)
            x2n = min(x2 + y, xi2)
            trans[(0, x2n, AT_1)] = \
                trans.get((0, x2n, AT_1), 0) + p
        return trans

    elif l == REP_2:
        # in the next unit, maintenance completed → machine 2 becomes healthy
        for y in range(xi1-x1+1):
            p = poisson_pmf(y,lam,xi1-x1)
            x1n = min(x1 + y, xi1)
            trans[(x1n, 0, AT_2)] = \
                trans.get((x1n, 0, AT_2), 0) + p
        return trans

    # ---------------- travel
    elif action == "travel_1":
        for y1 in range(xi1-x1+1):
            for y2 in range(xi2-x2+1):
                p = poisson_pmf(y1,lam,xi1-x1) * poisson_pmf(y2,lam,xi2-x2)
                x1n = min(x1 + y1, xi1)
                x2n = min(x2 + y2, xi2)
                trans[(x1n, x2n, AT_1)] = \
                    trans.get((x1n, x2n, AT_1), 0) + p
        return trans

    elif action == "travel_2":
        for y1 in range(xi1-x1+1):
            for y2 in range(xi2-x2+1):
                p = poisson_pmf(y1,lam,xi1-x1) * poisson_pmf(y2,lam,xi2-x2)
                x1n = min(x1 + y1, xi1)
                x2n = min(x2 + y2, xi2)
                trans[(x1n, x2n, AT_2)] = \
                    trans.get((x1n, x2n, AT_2), 0) + p
        return trans
    
    elif action == "travel_depot":
        for y1 in range(xi1-x1+1):
            for y2 in range(xi2-x2+1):
                p = poisson_pmf(y1,lam,xi1-x1) * poisson_pmf(y2,lam,xi2-x2)
                x1n = min(x1 + y1, xi1)
                x2n = min(x2 + y2, xi2)
                trans[(x1n, x2n, DEPOT)] = \
                    trans.get((x1n, x2n, DEPOT), 0) + p

    # ---------------- maintenance start
    elif action == "maintain_1":
        # preventive = 1 period
        if x1 < xi1:
            for y in range(xi2-x2+1):
                p = poisson_pmf(y,lam, xi2-x2)
                x2n = min(x2 + y, xi2)
                trans[(0, x2n, AT_1)] = \
                    trans.get((0, x2n, AT_1), 0) + p
        elif x1 == xi1:
            # corrective = 2 periods
            for y in range(xi2-x2+1):
                p = poisson_pmf(y,lam,xi2-x2)
                x2n = min(x2 + y, xi2)
                trans[(xi1, x2n, REP_1)] = \
                    trans.get((xi1, x2n, REP_1), 0) + p
        return trans

    elif action == "maintain_2":
        if x2 < xi2:
            for y in range(xi1-x1+1):
                p = poisson_pmf(y,lam,xi1-x1)
                x1n = min(x1 + y, xi1)
                trans[(x1n, 0, AT_2)] = \
                    trans.get((x1n, 0, AT_2), 0) + p
        elif x2==xi2:
            for y in range(xi1-x1+1):
                p = poisson_pmf(y,lam,xi1-x1)
                x1n = min(x1 + y, xi1)
                trans[(x1n, xi2, REP_2)] = \
                    trans.get((x1n, xi2, REP_2), 0) + p
        return trans

    elif action == "nothing":
        for y1 in range(xi1-x1+1):
            for y2 in range(xi2-x2+1):
                p = poisson_pmf(y1,lam,xi1-x1) * poisson_pmf(y2,lam,xi2-x2)
                x1n = min(x1 + y1, xi1)
                x2n = min(x2 + y2, xi2)
                trans[(x1n, x2n, l)] = \
                    trans.get((x1n, x2n, l), 0) + p
    return trans

# Assignment 2 — Task 3: Q-Learning with Linear Value Function Approximation

This notebook adapts the Q-Learning implementation from Task 1 to use **linear value function approximation (linear VFA)**. Instead of maintaining a full Q-table with one independent entry per feasible $(s,a)$ pair, we approximate the action-value function as:

$$\hat{q}(s, a, \mathbf{w}) = \mathbf{x}(s,a)^\top \mathbf{w}$$

where $\mathbf{x}(s,a) \in \mathbb{R}^{14}$ is a feature vector and $\mathbf{w} \in \mathbb{R}^{14}$ is a shared weight vector learned by stochastic gradient descent.

**Notebook structure:**
1. Policy Iteration oracle (ground-truth $V^*$)
2. Feature vector design and justification
3. Linear VFA Q-Learning: pseudocode and implementation
4. Training run with motivated hyperparameters and convergence analysis
5. Optimal policy extraction and comparison with Assignment 1 (PI)
6. Hyperparameter sensitivity study
7. Comparison with Task 1 (Q-Learning)

---
**Prerequisites:** Execute the Assignment 1 MDP definition cell before running this notebook.
Required in scope: `states`, `ACTIONS`, `feasible_actions`, `cost`, `transitions`,
`gamma`, `xi1`, `xi2`, `DEPOT`, `AT_1`, `AT_2`, `REP_1`, `REP_2`.

In [2]:
import numpy as np
import random
import math
import pandas as pd
import os
import json as _json
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ── Episode length (identical derivation to Task 1) ──────────────────────
def getLengthEpisode(gamma_val=0.9, r_max=7.0, error=1e-8):
    return int(np.ceil(
        np.log(error * (1.0 - gamma_val) / r_max) / np.log(gamma_val)
    ))

T_MAX = getLengthEpisode(gamma_val=gamma)
print(f"Episode length: T_MAX = {T_MAX} steps  (gamma={gamma}, R_max=7, error=1e-8)")

# ── Caches (same as Task 1) ───────────────────────────────────────────────
feasible_cache = {s: feasible_actions(s) for s in states}

cost_cache = {(s, a): cost(s, a) for s in states for a in feasible_cache[s]}

trans_cache = {}
for s in states:
    for a in feasible_cache[s]:
        td    = transitions(s, a)
        p_arr = np.array(list(td.values()), dtype=np.float64)
        p_arr /= p_arr.sum()
        trans_cache[(s, a)] = (list(td.keys()), p_arr)

print(f"Cached {len(trans_cache)} feasible (state, action) distributions")
print(f"States: {len(states)},  Actions: {len(ACTIONS)}")

Episode length: T_MAX = 216 steps  (gamma=0.9, R_max=7, error=1e-8)
Cached 369 feasible (state, action) distributions
States: 158,  Actions: 7


## 1 — Policy Iteration Oracle

Policy Iteration is run with full model knowledge to obtain $V^*(s)$ and $\pi^*(s)$.
These serve two purposes:
- **Oracle MSE**: objective measure of how close the learned $\hat{q}$ converges to $q^*$
- **Policy comparison**: verifying that linear VFA Q-Learning recovers the same decisions

In a real model-free setting this oracle would not be available.
It is used here because the model is known from Assignment 1.

In [3]:
V_PI     = {s: 0.0             for s in states}
policy_PI = {s: feasible_actions(s)[0] for s in states}


def _pi_eval(pol, V, tol=1e-10):
    while True:
        delta = 0.0
        for s in states:
            a     = pol[s]
            v_new = cost(s, a) + gamma * sum(
                p * V[sp] for sp, p in transitions(s, a).items()
            )
            delta = max(delta, abs(V[s] - v_new))
            V[s]  = v_new
        if delta < tol:
            break


def _pi_improve(pol, V):
    stable = True
    for s in states:
        old_a            = pol[s]
        best_a, best_q   = old_a, float("inf")
        for a in feasible_actions(s):
            q = cost(s, a) + gamma * sum(
                p * V[sp] for sp, p in transitions(s, a).items()
            )
            if q < best_q:
                best_q, best_a = q, a
        if best_a != old_a:
            pol[s] = best_a
            stable = False
    return stable


_pi_iters = 0
while True:
    _pi_iters += 1
    _pi_eval(policy_PI, V_PI)
    if _pi_improve(policy_PI, V_PI):
        break

print(f"PI converged in {_pi_iters} iterations")
print(f"V*(0,0,DEPOT) = {V_PI[(0, 0, DEPOT)]:.4f}")
print(f"V*(5,7,DEPOT) = {V_PI[(5, 7, DEPOT)]:.4f}  (worst degradation state)")

PI converged in 5 iterations
V*(0,0,DEPOT) = 2.4620
V*(5,7,DEPOT) = 16.8511  (worst degradation state)


## 2 — Feature Vector Design

### From Q-Table to Linear Approximation

In Task 1, the action-value function was stored as a lookup table: one independent number $Q(s,a)$ per feasible pair. This requires storing and updating $|\mathcal{S}| \times |\mathcal{A}|$ entries independently, with no sharing of information between states.

Linear VFA replaces this with:
$$\hat{q}(s, a, \mathbf{w}) = \mathbf{x}(s,a)^\top \mathbf{w}$$

where $\mathbf{x}(s,a) \in \mathbb{R}^{14}$ and $\mathbf{w} \in \mathbb{R}^{14}$ is a **single shared** weight vector. Updating $\mathbf{w}$ on one $(s,a)$ pair propagates information to all other pairs through the shared weights, enabling generalisation across similar states.

### Feature Vector Structure

The 14-dimensional feature vector for a pair $(s, a)$ with $s = (x_1, x_2, l)$ is:

| Index | Feature | Description |
|:---:|---|---|
| 0 | $x_1 / \xi_1$ | Machine 1 degradation, normalised to $[0,1]$ |
| 1 | $x_2 / \xi_2$ | Machine 2 degradation, normalised to $[0,1]$ |
| 2 | $\mathbf{1}(l = \text{DEPOT})$ | Location indicator |
| 3 | $\mathbf{1}(l = \text{AT\_1})$ | Location indicator |
| 4 | $\mathbf{1}(l = \text{AT\_2})$ | Location indicator |
| 5 | $\mathbf{1}(l = \text{REP\_1})$ | Location indicator |
| 6 | $\mathbf{1}(l = \text{REP\_2})$ | Location indicator |
| 7 | $\mathbf{1}(a = \text{nothing})$ | Action indicator |
| 8 | $\mathbf{1}(a = \text{travel\_1})$ | Action indicator |
| 9 | $\mathbf{1}(a = \text{travel\_2})$ | Action indicator |
| 10 | $\mathbf{1}(a = \text{travel\_depot})$ | Action indicator |
| 11 | $\mathbf{1}(a = \text{maintain\_1})$ | Action indicator |
| 12 | $\mathbf{1}(a = \text{maintain\_2})$ | Action indicator |
| 13 | $\mathbf{1}(a = \text{continue maintenance})$ | Action indicator |

**Why 14 and not one indicator per $(s,a)$ pair?**
If every feasible $(s,a)$ pair received its own indicator feature, the weight vector would have one entry per pair and $\hat{q}(s,a,\mathbf{w})$ would reduce exactly to the Q-table: no compression, no generalisation (the lecturer describes this degenerate case as "just renaming $V(s_i)$ as $w_i$").

The 14-dimensional design achieves genuine compression:
- **Continuous features** $x_1/\xi_1$, $x_2/\xi_2$ encode the degradation level numerically, allowing the model to generalise across adjacent degradation states (e.g., $x_1=3$ and $x_1=4$ receive similar feature values and similar Q-estimates).
- **Indicator features** for location and action maintain full expressiveness for the categorical components.
- Total parameters: **14** weights versus hundreds of independent Q-table entries.

### Update Rule

The gradient of $\hat{q}$ with respect to $\mathbf{w}$ is simply $\mathbf{x}(s,a)$ (derivative of a linear function). The SGD update therefore becomes:

$$\mathbf{w} \leftarrow \mathbf{w} + \alpha \cdot \underbrace{\bigl[r + \gamma \min_{a'} \hat{q}(s',a',\mathbf{w}) - \hat{q}(s,a,\mathbf{w})\bigr]}_{\delta \text{ (TD error)}} \cdot \mathbf{x}(s,a)$$

This is structurally identical to the Task 1 update, with $Q(s,a)$ replaced by $\mathbf{x}(s,a)^\top\mathbf{w}$ and the per-entry update replaced by a gradient step on the shared weight vector.

### Convergence Warning

From the convergence table in the lecture notes: **off-policy TD (Q-Learning) with linear function approximation has no guaranteed convergence**. The combination of bootstrapping (TD target uses its own estimate) and off-policy learning (behaviour policy differs from target policy) breaks the convergence proof. In practice, well-designed features and a decaying exploration rate often produce stable convergence, but this cannot be guaranteed theoretically.

In [4]:
# ── Feature vector ────────────────────────────────────────────────────────
N_FEATURES = 14

_LOC_IDX = {DEPOT: 2, AT_1: 3, AT_2: 4, REP_1: 5, REP_2: 6}
_ACT_IDX = {
    "nothing": 7,
    "travel_1": 8,
    "travel_2": 9,
    "travel_depot": 10,
    "maintain_1": 11,
    "maintain_2": 12,
    "continue maintenance": 13,
}


def feature_vector(state, action):
    """
    Build the 14-dimensional feature vector for a (state, action) pair.

    x(s, a) = [x1/xi1, x2/xi2,
               1(l=DEPOT), 1(l=AT_1), 1(l=AT_2), 1(l=REP_1), 1(l=REP_2),
               1(a=nothing), 1(a=travel_1), ..., 1(a=continue maintenance)]
    """
    x1, x2, l = state
    phi = np.zeros(N_FEATURES, dtype=np.float64)
    phi[0] = x1 / xi1          # normalised machine 1 degradation
    phi[1] = x2 / xi2          # normalised machine 2 degradation
    phi[_LOC_IDX[l]] = 1.0     # one-hot location
    phi[_ACT_IDX[action]] = 1.0  # one-hot action
    return phi


def q_hat(state, action, w):
    """Approximate action-value: q_hat(s,a,w) = x(s,a).T @ w"""""
    return float(np.dot(feature_vector(state, action), w))


def best_q_hat(state, w):
    """min_{a feasible} q_hat(s, a, w)  — cost minimisation."""""
    return min(q_hat(state, a, w) for a in feasible_cache[state])


def greedy_action_vfa(state, w):
    """argmin_{a feasible} q_hat(s, a, w) with random tie-breaking."""""
    acts = feasible_cache[state]
    vals = [q_hat(state, a, w) for a in acts]
    min_val = min(vals)
    tied = [a for a, v in zip(acts, vals) if v == min_val]
    return random.choice(tied)


print(f"Feature vector dimension: {N_FEATURES}")
print(f"Example phi((0,0,DEPOT), 'nothing'):")
print(feature_vector((0, 0, DEPOT), "nothing"))

Feature vector dimension: 14
Example phi((0,0,DEPOT), 'nothing'):
[0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]


## 3 — Linear VFA Q-Learning

### Pseudocode

```
INPUT  : K (max episodes), alpha_0 (constant learning rate), eps_0 (exploration scale)
DERIVE : T_max = ceil( log(error*(1-gamma)/R_max) / log(gamma) )
INIT   : w <- Uniform(-0.01, 0.01)^14     small random weights
         consec <- 0

FOR k = 0, 1, ..., K-1:
    w_prev <- w
    s      <- Uniform(S)                    random start state
    eps_k  <- eps_0 / (k + 1)              GLIE exploration rate

    FOR t = 0, 1, ..., T_max - 1:
        IF U[0,1] < eps_k:
            a <- uniform random from feasible_actions(s)           (explore)
        ELSE:
            a <- argmin_{a' feasible} x(s,a').T @ w, tie-break     (exploit)

        r  <- cost(s, a)
        s' <- sample from P(. | s, a)

        delta  <- r + gamma * min_{a'} x(s',a').T @ w - x(s,a).T @ w   (TD error)
        w      <- w + alpha_0 * delta * x(s, a)                          (gradient step)
        s      <- s'

    // End-of-episode convergence check
    dw_k <- max_i |w_i - w_prev_i|
    IF dw_k < tau for C consecutive episodes: STOP

RETURN w;   pi*(s) = argmin_{a feasible} x(s,a).T @ w
```

### Hyperparameter Justification

| Parameter | Value | Justification |
|---|---|---|
| `alpha_0` | constant (tuned) | Shared weight vector: per-visit Robbins-Monro per $(s,a)$ is not applicable. Constant $\alpha$ is standard for linear VFA SGD. |
| `epsilon_0` | 0.5 | Same GLIE schedule as Task 1/2: $\varepsilon_k = 0.5/(k+1) \to 0$, all pairs visited infinitely. |
| `T_max` | 216 | Same tail-bound derivation as Task 1: $\gamma^{T} R_{\max}/(1-\gamma) < 10^{-8}$. |
| $\tau$ | $10^{-4}$ | Same convergence threshold as Task 2; tighter than Task 1 ($10^{-3}$). |
| $C$ | 20 | 20 consecutive stable episodes, same as Task 1/2. |

**Why constant $\alpha$ instead of Robbins-Monro?**
In Task 1, each Q-table entry $Q(s,a)$ is updated independently, so a per-visit counter $N(s,a)$ gives each entry its own decaying rate. In Task 3, a single weight vector $\mathbf{w}$ is updated by every step regardless of which $(s,a)$ was visited. A per-pair counter would not correspond to the number of times each weight was updated, making the Robbins-Monro schedule ill-defined. A constant $\alpha$ is the standard choice for linear VFA and controls the step size of each gradient update directly.

In [5]:
def qLearning_linear_VFA(
    n_episodes,
    alpha_0=0.005,
    epsilon_0=0.5,
    V_oracle=None,
    mse_interval=200,
    dw_tau=1e-4,
    consec_threshold=20,
    seed=None,
):
    """
    Q-Learning with Linear Value Function Approximation.

    Approximation : q_hat(s, a, w) = x(s,a).T @ w
    Update rule   : w <- w + alpha * delta * x(s,a)
                    delta = r + gamma * min_{a'} q_hat(s',a',w) - q_hat(s,a,w)

    Parameters
    ----------
    n_episodes        : maximum number of training episodes
    alpha_0           : constant learning rate (shared across all weight updates)
    epsilon_0         : GLIE exploration scale; eps_k = epsilon_0 / (k+1)
    V_oracle          : dict {state: V*(s)} from PI for oracle MSE tracking
    mse_interval      : episodes between oracle MSE checkpoints
    dw_tau            : convergence threshold on max|Delta w|
    consec_threshold  : stop when max|Delta w| < dw_tau for this many consecutive episodes
    seed              : random seed for reproducibility

    Returns
    -------
    w               : ndarray (14,), learned weight vector
    episode_costs   : list[float], total undiscounted cost per episode
    dw_history      : list[float], max|Delta w| per episode
    mse_checkpoints : list[(episode, mse)]
    conv_episode    : int, episode at which convergence was declared
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    w = np.random.uniform(-0.01, 0.01, size=N_FEATURES)

    episode_costs   = []
    dw_history      = []
    mse_checkpoints = []
    conv_episode    = n_episodes
    consec_count    = 0

    for k in tqdm(range(n_episodes), desc="Linear VFA Q-Learning", leave=True):

        w_before = w.copy()
        s        = random.choice(states)
        eps_k    = epsilon_0 / (k + 1)
        ep_cost  = 0.0

        for _ in range(T_MAX):
            if random.random() < eps_k:
                a = random.choice(feasible_cache[s])      # explore
            else:
                a = greedy_action_vfa(s, w)               # exploit

            r        = cost_cache[(s, a)]
            ep_cost += r

            next_states, probs = trans_cache[(s, a)]
            s_new = next_states[np.random.choice(len(next_states), p=probs)]

            # TD error
            q_cur   = q_hat(s, a, w)
            q_next  = best_q_hat(s_new, w)
            delta   = r + gamma * q_next - q_cur

            # Gradient step: w <- w + alpha * delta * x(s,a)
            phi = feature_vector(s, a)
            w  += alpha_0 * delta * phi

            s = s_new

        episode_costs.append(ep_cost)

        # Convergence metric: max component change in weight vector
        dw = float(np.max(np.abs(w - w_before)))
        dw_history.append(dw)

        # Oracle MSE
        if V_oracle is not None and k % mse_interval == 0:
            V_approx = np.array([best_q_hat(s, w) for s in states])
            V_ref    = np.array([V_oracle[s]       for s in states])
            mse_checkpoints.append((k, float(np.mean((V_approx - V_ref) ** 2))))

        # Consecutive convergence check
        if dw < dw_tau:
            consec_count += 1
            if consec_count >= consec_threshold:
                conv_episode = k
                break
        else:
            consec_count = 0

    return w, episode_costs, dw_history, mse_checkpoints, conv_episode

## 4 — Training Run

The main run uses the best hyperparameters selected from the sensitivity study in Section 6:
$\alpha_0 = 0.005$, $\varepsilon_0 = 0.5$, $K = 80\,000$ episodes.

The weight vector is initialised with small uniform random values $\sim U(-0.01, 0.01)$ to break symmetry (zero initialisation causes all features to receive the same gradient in the first episode, slowing convergence).

In [ ]:
w_main, costs_vfa, dw_vfa, mse_vfa, conv_vfa = qLearning_linear_VFA(
    n_episodes       = 80_000,
    alpha_0          = 0.005,
    epsilon_0        = 0.5,
    V_oracle         = V_PI,
    mse_interval     = 200,
    dw_tau           = 1e-3,
    consec_threshold = 20,
    seed             = 42,
)

print(f"\nConverged at episode  : {conv_vfa}")
print(f"Episodes run          : {len(costs_vfa)}")
print(f"Mean cost (last 500)  : {np.mean(costs_vfa[-500:]):.4f}")
print(f"Final max|Delta w|    : {dw_vfa[-1]:.2e}")
print(f"Learned weights w     :\n{np.round(w_main, 4)}")

Linear VFA Q-Learning:   0%|          | 0/80000 [00:00<?, ?it/s]

## 5 — Convergence Analysis

**Convergence metric.** At the end of each episode $k$, the maximum absolute change in any component of $\mathbf{w}$ is computed:

$$\delta_k^w = \max_{i} \left|w_i^{(k)} - w_i^{(k-1)}\right|$$

**Stopping rule.** Training terminates when $\delta_k^w < \tau = 10^{-4}$ for $C = 20$ consecutive episodes, or after $K = 80\,000$ episodes.

**Rationale.** $\delta_k^w$ directly measures the stability of the weight vector that determines both $\hat{q}$ and the greedy policy. When $\mathbf{w}$ stabilises, $\pi^*(s) = \arg\min_a \mathbf{x}(s,a)^\top \mathbf{w}$ no longer changes. This is the natural analogue of the max$|\Delta Q|$ criterion used in Task 1/2.

In [ ]:
window = 500
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Left: per-episode cost + rolling mean
ax = axes[0]
sm = np.convolve(costs_vfa, np.ones(window) / window, mode="valid")
ax.plot(costs_vfa, color="steelblue", alpha=0.12, linewidth=0.4)
ax.plot(np.arange(window - 1, len(costs_vfa)), sm,
        color="steelblue", linewidth=1.8, label=f"Rolling mean (w={window})")
ax.axvline(conv_vfa, color="crimson", linestyle=":", linewidth=1.2,
           label=f"Episode {conv_vfa}")
ax.set_xlabel("Episode")
ax.set_ylabel("Total cost per episode")
ax.set_title("Linear VFA: Episode Cost  (alpha=0.005, eps0=0.5)")
ax.legend(fontsize=8)

# Centre: max|Delta w| log scale
ax = axes[1]
roll_min_dw = [np.min(dw_vfa[max(0, j - window):j + 1]) for j in range(len(dw_vfa))]
ax.semilogy(dw_vfa, alpha=0.12, color="darkorange", linewidth=0.4)
ax.semilogy(roll_min_dw, color="darkorange", linewidth=1.8,
            label="Rolling min of max|Delta w|")
ax.axhline(1e-4, color="green", linestyle="--", linewidth=1.2,
           label="Threshold tau = 1e-4")
ax.axvline(conv_vfa, color="crimson", linestyle=":", linewidth=1.2,
           label=f"Episode {conv_vfa}")
ax.set_xlabel("Episode")
ax.set_ylabel("max|Delta w|")
ax.set_title("Convergence Metric  (alpha=0.005, eps0=0.5)")
ax.legend(fontsize=8)

# Right: oracle MSE
if mse_vfa:
    ep_idx, mse_vals = zip(*mse_vfa)
    ax = axes[2]
    ax.semilogy(ep_idx, mse_vals, color="seagreen", linewidth=1.5,
                marker=".", markersize=2, label="Oracle MSE")
    ax.axvline(conv_vfa, color="crimson", linestyle=":", linewidth=1.2,
               label=f"Episode {conv_vfa}")
    ax.set_xlabel("Episode")
    ax.set_ylabel("MSE (log scale)")
    ax.set_title("Oracle MSE: min_a q_hat vs V*(s)")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("fig_vfa_convergence.pdf", bbox_inches="tight")
plt.show()

**Convergence interpretation.** The left panel shows the per-episode cost and its rolling mean (window = 500). The rolling mean stabilises, indicating that the behaviour policy has reached a near-stationary regime. The centre panel shows max$|\Delta\mathbf{w}|$ on a log scale. If it drops below $\tau = 10^{-4}$ and remains there for 20 consecutive episodes, training stops early; otherwise the algorithm runs the full episode budget. The right panel shows the oracle MSE: the mean squared difference between $\min_a \hat{q}(s,a,\mathbf{w})$ and $V^*(s)$ across all states. A decreasing oracle MSE confirms that the weight vector is converging towards the true optimal value function.

## 6 — Optimal Policy

The greedy policy is extracted as:

$$\pi^*(s) = \arg\min_{a \in \mathcal{A}(s)} \mathbf{x}(s,a)^\top \mathbf{w}$$

with random tie-breaking. States in REP\_1 and REP\_2 are excluded (action is forced: "continue maintenance"). The tables below show the recommended action for each degradation-level pair $(x_1, x_2)$ by engineer location, and compare to the PI oracle from Assignment 1.

In [ ]:
def extract_policy_vfa(w):
    """Extract greedy policy from weight vector with random tie-breaking."""""
    return {s: greedy_action_vfa(s, w) for s in states}


def print_policy_table(policy_dict, location, title=""):
    """Display policy as a DataFrame (rows=x1, cols=x2) for a given location."""""
    rows = []
    for x1 in range(xi1 + 1):
        row = []
        for x2 in range(xi2 + 1):
            s = (x1, x2, location)
            row.append(policy_dict.get(s, "N/A"))
        rows.append(row)
    df = pd.DataFrame(
        rows,
        index   = [f"x1={i}" for i in range(xi1 + 1)],
        columns = [f"x2={j}" for j in range(xi2 + 1)],
    )
    print(f"\n{title}")
    print(df.to_string())


policy_VFA = extract_policy_vfa(w_main)
loc_names  = {DEPOT: "DEPOT", AT_1: "AT Machine 1", AT_2: "AT Machine 2"}

print("=" * 60)
print("LINEAR VFA Q-LEARNING POLICY  (alpha=0.005, eps0=0.5)")
print("=" * 60)
for loc, name in loc_names.items():
    print_policy_table(policy_VFA, loc, f"VFA — {name}")

print("\n" + "=" * 60)
print("POLICY ITERATION ORACLE  (Assignment 1 ground truth)")
print("=" * 60)
for loc, name in loc_names.items():
    print_policy_table(policy_PI, loc, f"PI  — {name}")

# ── Match statistics ──────────────────────────────────────────────────────
total_nf = sum(1 for s in states if s[2] not in (REP_1, REP_2))
exact    = [s for s in states
            if s[2] not in (REP_1, REP_2) and policy_VFA[s] == policy_PI[s]]
mismatch = [s for s in states
            if s[2] not in (REP_1, REP_2) and policy_VFA[s] != policy_PI[s]]
match_pct = 100.0 * len(exact) / total_nf

print(f"\nNon-forced states     : {total_nf}")
print(f"Match with PI oracle  : {len(exact)}  ({match_pct:.1f} %)")
print(f"Mismatches            : {len(mismatch)}")

# Classify mismatches by approximate Q-value gap
THRESH = 0.05
near_tie_vfa, true_mismatch_vfa = [], []
for s in mismatch:
    a_pi  = policy_PI[s]
    a_vfa = policy_VFA[s]
    q_vfa_chosen = q_hat(s, a_vfa, w_main)
    q_pi_action  = q_hat(s, a_pi,  w_main)
    gap  = abs(q_vfa_chosen - q_pi_action)
    if gap < THRESH:
        near_tie_vfa.append((s, a_pi, a_vfa, gap))
    else:
        true_mismatch_vfa.append((s, a_pi, a_vfa, gap))

print(f"Near-tie (gap < {THRESH}) : {len(near_tie_vfa)}")
print(f"True mismatches       : {len(true_mismatch_vfa)}")

if near_tie_vfa:
    print("\nNear-tie mismatches:")
    for s, a_pi, a_vfa, gap in near_tie_vfa:
        print(f"  {s}  PI={a_pi:<22s}  VFA={a_vfa:<22s}  gap={gap:.4f}")

if true_mismatch_vfa:
    print("\nTrue mismatches:")
    for s, a_pi, a_vfa, gap in true_mismatch_vfa:
        print(f"  {s}  PI={a_pi:<22s}  VFA={a_vfa:<22s}  gap={gap:.4f}")

## 7 — Hyperparameter Sensitivity Study

A grid search over $\alpha_0 \in \{0.001, 0.005, 0.01\}$ and $\varepsilon_0 \in \{0.5, 1.0\}$ (6 configurations) is run for $K = 80\,000$ episodes each. Results are saved to `hyperparam_vfa_results.json` after each configuration so the study is fully resumable.

The exploration schedule $\varepsilon_0$ follows the same GLIE argument as Task 1: $\varepsilon_k = \varepsilon_0/(k+1) \to 0$, ensuring every state-action pair is visited infinitely often.

For the learning rate $\alpha_0$: too large a value causes oscillation in $\mathbf{w}$ (the gradient step overshoots the optimum); too small a value slows learning. The grid covers one order of magnitude around $0.005$.

In [ ]:
VFA_RESULTS_FILE = "hyperparam_vfa_results.json"

CONFIGS_VFA = [
    {"alpha_0": 0.001, "epsilon_0": 0.5,  "label": "a=0.001, e=0.5"},
    {"alpha_0": 0.001, "epsilon_0": 1.0,  "label": "a=0.001, e=1.0"},
    {"alpha_0": 0.005, "epsilon_0": 0.5,  "label": "a=0.005, e=0.5"},
    {"alpha_0": 0.005, "epsilon_0": 1.0,  "label": "a=0.005, e=1.0"},
    {"alpha_0": 0.01,  "epsilon_0": 0.5,  "label": "a=0.010, e=0.5"},
    {"alpha_0": 0.01,  "epsilon_0": 1.0,  "label": "a=0.010, e=1.0"},
]

if os.path.exists(VFA_RESULTS_FILE):
    with open(VFA_RESULTS_FILE) as _f:
        vfa_study = _json.load(_f)
    _done = {r["label"] for r in vfa_study}
    print(f"Loaded {len(vfa_study)} saved result(s). Skipping: {_done}")
else:
    vfa_study = []
    _done     = set()

_total_nrep = sum(1 for s in states if s[2] not in (REP_1, REP_2))

for cfg in CONFIGS_VFA:
    if cfg["label"] in _done:
        print(f"[skip] {cfg['label']}")
        continue
    print(f"Running: {cfg['label']}")

    w_c, costs_c, dw_c, _, conv_c = qLearning_linear_VFA(
        n_episodes       = 80_000,
        alpha_0          = cfg["alpha_0"],
        epsilon_0        = cfg["epsilon_0"],
        dw_tau           = 1e-4,
        consec_threshold = 20,
        seed             = 42,
    )

    pol_c   = extract_policy_vfa(w_c)
    n_match = sum(1 for s in states
                  if s[2] not in (REP_1, REP_2) and pol_c[s] == policy_PI[s])
    match_pct  = 100.0 * n_match / _total_nrep
    final_cost = (float(np.mean(costs_c[-500:]))
                  if len(costs_c) >= 500 else float(np.mean(costs_c)))

    # Bias: min_a q_hat(s,w) vs V*(s)
    errs_c = [float(best_q_hat(s, w_c)) - V_PI[s]
              for s in states if s[2] not in (REP_1, REP_2)]

    result = {
        "label"     : cfg["label"],
        "conv"      : conv_c,
        "costs"     : costs_c,
        "dw"        : dw_c,
        "match_pct" : match_pct,
        "final_cost": final_cost,
        "mean_bias" : float(np.mean(errs_c)),
        "w"         : w_c.tolist(),
    }
    vfa_study.append(result)
    with open(VFA_RESULTS_FILE, "w") as _f:
        _json.dump(vfa_study, _f)

    print(f"  conv={conv_c:>6d}  match={match_pct:.1f}%  "
          f"cost={final_cost:.2f}  bias={np.mean(errs_c):+.4f}")
    print()

print("\nVFA hyperparameter study complete.")

In [ ]:
import matplotlib.cm as _cm

_cmap  = plt.colormaps["tab10"].resampled(max(len(vfa_study), 1))
_cols  = [_cmap(i) for i in range(len(vfa_study))]
window = 300

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: rolling mean cost per config
ax = axes[0]
for r, c in zip(vfa_study, _cols):
    sm = np.convolve(r["costs"], np.ones(window) / window, mode="valid")
    ax.plot(np.arange(window - 1, len(r["costs"])), sm,
            color=c, linewidth=1.5, label=r["label"])
ax.set_xlabel("Episode")
ax.set_ylabel(f"Rolling mean cost (w={window})")
ax.set_title("Episode Cost — Linear VFA Configurations")
ax.legend(fontsize=8, ncol=2, loc="upper right")

# Right: convergence episode bar chart
ax = axes[1]
labels = [r["label"] for r in vfa_study]
convs  = [r["conv"]  for r in vfa_study]
bars   = ax.bar(range(len(labels)), convs, color=_cols, alpha=0.85, edgecolor="white")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("Convergence episode")
ax.set_title("Episodes to Convergence — Linear VFA")
for bar, val in zip(bars, convs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            str(val), ha="center", va="bottom", fontsize=7)

plt.tight_layout()
plt.savefig("fig_vfa_hyperparam_study.pdf", bbox_inches="tight")
plt.show()

# Summary table
import pandas as _pd
df_vfa = _pd.DataFrame([{
    "Configuration"   : r["label"],
    "Convergence ep." : r["conv"],
    "Policy match (%)": f"{r['match_pct']:.1f}",
    "Final mean cost" : f"{r['final_cost']:.3f}",
    "Mean bias"       : f"{r['mean_bias']:+.4f}",
} for r in vfa_study])
print(df_vfa.to_string(index=False))

## 8 — Task 1 vs Task 3: Summary of Differences

| Property | Q-Learning (Task 1) | Linear VFA (Task 3) |
|---|---|---|
| Value function | Q-table: $|S|\times|A|$ independent entries | $\hat{q}(s,a,\mathbf{w}) = \mathbf{x}(s,a)^\top\mathbf{w}$: 14 shared weights |
| Parameters | One per feasible $(s,a)$ | 14 total, regardless of state/action count |
| Generalisation | None — each $(s,a)$ updated independently | Yes — updating $\mathbf{w}$ propagates to all $(s,a)$ |
| Learning rate | Robbins-Monro: $\alpha_k = 1/N(s,a)^\omega$ | Constant $\alpha_0$ (gradient step size) |
| Update scope | Local: only $Q(s,a)$ changes | Global: all $\hat{q}$ values change via $\mathbf{w}$ |
| Convergence guarantee | Yes, under GLIE + Robbins-Monro | No — off-policy TD with linear VFA is not guaranteed to converge |
| Initialisation | $Q(s,a) = 0$ for feasible, $+\infty$ for infeasible | $\mathbf{w} \sim U(-0.01, 0.01)$; infeasibility handled via `feasible_cache` |
| Convergence metric | max$|\Delta Q|$ per episode | max$|\Delta\mathbf{w}|$ per episode |
| Dominant failure mode | Slow convergence (many independent entries to learn) | Instability (gradient steps can diverge with large $\alpha$) |

**Key observation.** Task 3 imposes a strong parametric constraint: $\hat{q}(s,a,\mathbf{w})$ must be a linear function of 14 features for all $(s,a)$ pairs. This means the true $q^*$ can only be recovered exactly if it lies in the span of the chosen feature vectors — which is generally not the case. The policy match percentage and oracle MSE therefore measure how well the 14-dimensional linear approximation can represent $q^*$, not just whether the learning algorithm has converged.